# Project 4b — Module 5: Statistical Inference
## Lesson 2: Probability & Sampling Design

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phase** | 2 — Data Understanding |
| **Module** | 5 — Statistical Inference (Alkemy Bootcamp) |
| **Dataset** | PequeShop — customers_final.csv + transactions_final.csv |
| **Date** | 2026-03 |

---

> **Executive Summary:**  
> This notebook applies probability theory to PequeShop's customer data.
> It defines the sample space and key events (churn, NPS segments, platform),
> calculates basic and conditional probabilities, constructs a probability tree
> (NPS x retargeting segment), and demonstrates three sampling strategies.
> All calculations directly connect to the four hypotheses defined in notebook 01.


## Table of Contents

1. [CRISP-DM Phase 2 — Data Understanding](#1-crisp-dm-phase-2--data-understanding)
2. [Load Data & Verify Structure](#2-load-data--verify-structure)
3. [Sample Space & Event Definition](#3-sample-space--event-definition)
4. [Basic Probabilities & Complements](#4-basic-probabilities--complements)
5. [Intersection, Union & Conditional Probability](#5-intersection-union--conditional-probability)
6. [Probability Tree: NPS x Retargeting Segment](#6-probability-tree-nps-x-retargeting-segment)
7. [Sampling Design](#7-sampling-design)
8. [Probability Summary — Connection to Hypotheses](#8-probability-summary--connection-to-hypotheses)
9. [LEAN Filter — Waste Elimination Review](#9-lean-filter--waste-elimination-review)
10. [Decisions Log — Lesson 2](#10-decisions-log--lesson-2)
11. [Next Steps — Lesson 3 Preview](#11-next-steps--lesson-3-preview)


---
## 1. CRISP-DM Phase 2 — Data Understanding

### 1.1 Objective

In this phase we explore the data through a **probabilistic lens** — not just
descriptive statistics (already done in project-3), but formal probability
calculations that will feed directly into the hypothesis tests of notebook 06.

### 1.2 Lean Filter for This Phase

> *Only calculate probabilities that inform at least one of the four hypotheses
> (H1–H4) or the sampling design. Everything else is waste.*

| Probability | Hypothesis Connected |
|-------------|---------------------|
| P(Dormant) = churn rate | H3 — churn vs 30% benchmark |
| P(Promoter), P(Passive), P(Detractor) | H4 — NPS effect on avg_ticket |
| P(NPS \| Segment) | H4 — conditional behavior by segment |
| P(platform) | H2 — MercadoLibre vs Shopify |


In [ ]:
# ===== Environment Setup =====
import warnings
warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ===== Plot Style =====
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')

# ===== Reproducibility =====
np.random.seed(42)

# ===== Paths =====
DATA_PROCESSED  = Path('../data/processed')
REPORTS_FIGURES = Path('../reports/figures')
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Data path   : {DATA_PROCESSED}')
print(f'Figures path: {REPORTS_FIGURES}')

---
## 2. Load Data & Verify Structure


In [ ]:
# ===== Load Datasets =====
df_customers     = pd.read_csv(DATA_PROCESSED / 'customers_final.csv')
df_transactions  = pd.read_csv(DATA_PROCESSED / 'transactions_final.csv')

print('=' * 55)
print('DATASET SUMMARY')
print('=' * 55)
print(f'customers_final   : {df_customers.shape[0]:>4} rows x {df_customers.shape[1]} cols')
print(f'transactions_final: {df_transactions.shape[0]:>4} rows x {df_transactions.shape[1]} cols')
print()
print('customers columns :', list(df_customers.columns))
print('transactions cols :', list(df_transactions.columns))

In [ ]:
# ===== Quick Data Quality Check =====
print('Null values — customers_final:')
print(df_customers.isnull().sum()[df_customers.isnull().sum() > 0])
print()
print('Null values — transactions_final:')
print(df_transactions.isnull().sum()[df_transactions.isnull().sum() > 0])
print()
print('retargeting_segment distribution:')
print(df_customers['retargeting_segment'].value_counts())
print()
print('nps_category distribution:')
print(df_customers['nps_category'].value_counts())

---
## 3. Sample Space & Event Definition

Before calculating probabilities, we formally define the **sample space (Omega)**
and the events of interest.

| Symbol | Event Definition | Source Column |
|--------|-----------------|---------------|
| **Omega** | All active PequeShop customers | customers_final (n=392) |
| **A** | Customer is Dormant (churned) | retargeting_segment == 'Dormant' |
| **B** | Customer is a Promoter | nps_category == 'Promoter' |
| **C** | Customer is At-Risk | retargeting_segment == 'At-Risk' |
| **D** | Customer acquired via Shopify | platform == 'Shopify' |
| **E** | Customer is Active | retargeting_segment == 'Active' |


In [ ]:
# ===== Sample Space =====
n_total     = len(df_customers)

# Retargeting segments
n_dormant   = (df_customers['retargeting_segment'] == 'Dormant').sum()
n_at_risk   = (df_customers['retargeting_segment'] == 'At-Risk').sum()
n_active    = (df_customers['retargeting_segment'] == 'Active').sum()

# NPS categories
n_promoter  = (df_customers['nps_category'] == 'Promoter').sum()
n_passive   = (df_customers['nps_category'] == 'Passive').sum()
n_detractor = (df_customers['nps_category'] == 'Detractor').sum()

print(f'Sample space |Omega| = {n_total} customers')
print()
print('Retargeting segments:')
print(f'  Active   (E): {n_active:>3}  ({n_active/n_total:.1%})')
print(f'  At-Risk  (C): {n_at_risk:>3}  ({n_at_risk/n_total:.1%})')
print(f'  Dormant  (A): {n_dormant:>3}  ({n_dormant/n_total:.1%})')
print()
print('NPS categories:')
print(f'  Promoter (B): {n_promoter:>3}  ({n_promoter/n_total:.1%})')
print(f'  Passive     : {n_passive:>3}  ({n_passive/n_total:.1%})')
print(f'  Detractor   : {n_detractor:>3}  ({n_detractor/n_total:.1%})')

---
## 4. Basic Probabilities & Complements


In [ ]:
# ===== Basic Probabilities =====
P_A       = n_dormant   / n_total   # P(Dormant)
P_B       = n_promoter  / n_total   # P(Promoter)
P_C       = n_at_risk   / n_total   # P(At-Risk)
P_E       = n_active    / n_total   # P(Active)
P_passive = n_passive   / n_total
P_detractor = n_detractor / n_total

print('=' * 50)
print('BASIC PROBABILITIES')
print('=' * 50)
print(f'P(A) = P(Dormant)    = {P_A:.4f}  ({P_A*100:.1f}%)')
print(f'P(B) = P(Promoter)   = {P_B:.4f}  ({P_B*100:.1f}%)')
print(f'P(C) = P(At-Risk)    = {P_C:.4f}  ({P_C*100:.1f}%)')
print(f'P(E) = P(Active)     = {P_E:.4f}  ({P_E*100:.1f}%)')
print()
print('COMPLEMENTS')
print(f'P(A^c) = P(not Dormant)   = {1-P_A:.4f}  ({(1-P_A)*100:.1f}%)')
print(f'P(B^c) = P(not Promoter)  = {1-P_B:.4f}  ({(1-P_B)*100:.1f}%)')
print()
print('SANITY CHECKS')
print(f'Segments sum to 1: {P_A + P_C + P_E:.4f}')
print(f'NPS cats sum to 1: {P_B + P_passive + P_detractor:.4f}')

---
## 5. Intersection, Union & Conditional Probability


In [ ]:
# ===== Intersection: P(Dormant AND Promoter) =====
# Business question: how many churned customers were once promoters?
n_dormant_promoter = (
    (df_customers['retargeting_segment'] == 'Dormant') &
    (df_customers['nps_category'] == 'Promoter')
).sum()
P_A_and_B = n_dormant_promoter / n_total

# Union: P(Dormant OR Promoter)
P_A_or_B = P_A + P_B - P_A_and_B

# Conditional: P(Promoter | Active)
df_active = df_customers[df_customers['retargeting_segment'] == 'Active']
P_B_given_E = (df_active['nps_category'] == 'Promoter').sum() / n_active

# Conditional: P(Dormant | Detractor)
df_detractor = df_customers[df_customers['nps_category'] == 'Detractor']
P_A_given_detractor = (
    df_detractor['retargeting_segment'] == 'Dormant'
).sum() / n_detractor

print('=' * 55)
print('INTERSECTION, UNION & CONDITIONAL PROBABILITY')
print('=' * 55)
print(f'P(A AND B) = P(Dormant AND Promoter)  = {P_A_and_B:.4f} ({P_A_and_B*100:.1f}%)')
print(f'P(A OR B)  = P(Dormant OR Promoter)   = {P_A_or_B:.4f} ({P_A_or_B*100:.1f}%)')
print()
print(f'P(Promoter | Active)      = {P_B_given_E:.4f} ({P_B_given_E*100:.1f}%)')
print(f'P(Dormant  | Detractor)   = {P_A_given_detractor:.4f} ({P_A_given_detractor*100:.1f}%)')
print()
print('Business Interpretation:')
print(f'  -> {P_A_given_detractor*100:.1f}% of Detractors have already churned (Dormant)')
print(f'  -> {P_B_given_E*100:.1f}% of Active customers are Promoters')
print(f'  -> Detractors are a high-priority retention target (churn risk + dissatisfaction)')

---
## 6. Probability Tree: NPS x Retargeting Segment

Visualizing P(NPS category | Retargeting segment) across all combinations.
This is the probability tree required by Lesson 2.


In [ ]:
# ===== Contingency Table =====
ct = pd.crosstab(
    df_customers['retargeting_segment'],
    df_customers['nps_category'],
    margins=True
)
print('Contingency table (counts):')
display(ct)
print()

# P(NPS | Segment)
ct_pct = pd.crosstab(
    df_customers['retargeting_segment'],
    df_customers['nps_category'],
    normalize='index'
).round(4)
print('P(NPS category | Segment):')
display(ct_pct)

In [ ]:
# ===== Probability Tree + Complementary Analysis =====
def plot_probability_tree(df: pd.DataFrame, ct_pct: pd.DataFrame) -> None:
    """Visualizes NPS x Segment conditional probabilities.

    Figure 1: Probability tree (nodes + arrows) — P(Segment) and P(NPS | Segment)
    Figure 2: Heatmap + stacked bar — complementary conditional probability analysis

    Args:
        df: pd.DataFrame — full customers dataset
        ct_pct: pd.DataFrame — P(NPS | Segment) conditional probability table

    Returns:
        None
    """
    import matplotlib.patches as mpatches

    # ── Marginal probabilities P(Segment) ──
    seg_counts = df['retargeting_segment'].value_counts(normalize=True)
    segments   = ['Active', 'At Risk', 'Dormant']
    seg_probs  = {s: seg_counts.get(s, 0) for s in segments}
    nps_order  = ['Promoter', 'No Response', 'Passive', 'Detractor']
    nps_colors = {
        'Promoter':    '#2ecc71',
        'No Response': '#f39c12',
        'Passive':     '#e74c3c',
        'Detractor':   '#c0392b'
    }

    # ── FIGURE 1: Probability Tree ──────────────────────────────
    fig1, ax = plt.subplots(figsize=(18, 10))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    fig1.patch.set_facecolor('white')
    ax.set_title(
        'Probability Tree: Retargeting Segment → NPS Category',
        fontsize=14, fontweight='bold', pad=20
    )

    # Root node
    root_x, root_y = 0.8, 5.0
    root_box = dict(boxstyle='round,pad=0.5', facecolor='#2980B9',
                    edgecolor='#1a5276', lw=2)
    ax.text(root_x, root_y, 'Customers\n(Ω)',
            ha='center', va='center', fontsize=11,
            fontweight='bold', color='white', bbox=root_box)

    # Segment node y-positions
    seg_y = {'Active': 8.2, 'At Risk': 5.0, 'Dormant': 1.8}
    seg_x = 3.8

    for seg in segments:
        prob  = seg_probs[seg]
        sy    = seg_y[seg]

        # Arrow root → segment node
        ax.annotate('', xy=(seg_x - 0.5, sy), xytext=(root_x + 0.55, root_y),
                    arrowprops=dict(arrowstyle='->', color='#2980B9', lw=2.0))

        # Probability label on arrow
        mid_x = (root_x + seg_x) / 2 - 0.2
        mid_y = (root_y + sy) / 2
        ax.text(mid_x, mid_y, f'P({seg[:2]})={prob:.2f}',
                fontsize=8.5, color='#2980B9', style='italic', ha='center')

        # Segment node box
        seg_box = dict(boxstyle='round,pad=0.4', facecolor='#5DADE2',
                       edgecolor='#2980B9', lw=1.5)
        ax.text(seg_x, sy, f'{seg}\nP={prob:.2f}',
                ha='center', va='center', fontsize=9.5,
                fontweight='bold', color='white', bbox=seg_box)

        # NPS leaf nodes
        nps_vals = ct_pct.loc[seg] if seg in ct_pct.index else {}
        n_nps    = len(nps_order)
        spread   = 2.8
        leaf_x   = 7.8

        for i, nps in enumerate(nps_order):
            prob_nps = nps_vals.get(nps, 0) if hasattr(nps_vals, 'get') else 0
            ly = sy + spread * (0.5 - i / (n_nps - 1))

            color = nps_colors.get(nps, '#888')
            ax.annotate('', xy=(leaf_x - 0.3, ly), xytext=(seg_x + 0.5, sy),
                        arrowprops=dict(arrowstyle='->', color=color, lw=1.8))

            # Probability on arrow
            ax.text((seg_x + leaf_x) / 2 + 0.1, (sy + ly) / 2,
                    f'{prob_nps:.1%}',
                    fontsize=8, color=color, style='italic', ha='center')

            # Leaf label
            ax.text(leaf_x, ly,
                    f'{nps}\nP({nps[:2]}|{seg[:2]})={prob_nps:.2f}',
                    ha='left', va='center', fontsize=8.5,
                    fontweight='bold', color=color)

    plt.tight_layout()
    out1 = REPORTS_FIGURES / '02_probability_tree_nps_segment.png'
    fig1.savefig(out1, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {out1}')

    # ── FIGURE 2: Heatmap + Stacked Bar (complementary) ────────
    fig2, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig2.suptitle(
        'Conditional Probability: P(NPS | Retargeting Segment)',
        fontsize=13, fontweight='bold'
    )

    sns.heatmap(
        ct_pct, annot=True, fmt='.1%',
        cmap='Blues', ax=axes[0], linewidths=0.5
    )
    axes[0].set_title('P(NPS | Segment) — Heatmap', fontweight='bold')
    axes[0].set_xlabel('NPS Category')
    axes[0].set_ylabel('Retargeting Segment')

    ct_pct.plot(
        kind='bar', stacked=True, ax=axes[1],
        color=[nps_colors.get(n, '#888') for n in ct_pct.columns],
        edgecolor='white'
    )
    axes[1].set_title('NPS Distribution by Segment — Stacked Bar', fontweight='bold')
    axes[1].set_xlabel('Retargeting Segment')
    axes[1].set_ylabel('Proportion')
    axes[1].legend(title='NPS Category', bbox_to_anchor=(1.05, 1))
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    out2 = REPORTS_FIGURES / '02_conditional_probability_heatmap.png'
    fig2.savefig(out2, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {out2}')


plot_probability_tree(df_customers, ct_pct)


---
## 7. Sampling Design

**Sampling type applied:** Census (exhaustive — all active customers included).

Since the dataset contains **all** active PequeShop customers (n=392),
we are working with a census — sampling error is eliminated.

To satisfy Module 5 requirements, we demonstrate three sampling strategies
and compare their estimates of `avg_ticket` against the census mean.


In [ ]:
# ===== Sampling Strategies Demonstration =====
np.random.seed(42)
n_sample = 100  # minimum required by M5 consigna

# 1. Simple random sampling
sample_random = df_customers.sample(n=n_sample, random_state=42)

# 2. Stratified sampling by retargeting_segment
sample_stratified = (
    df_customers
    .groupby('retargeting_segment', group_keys=False)
    .apply(lambda x: x.sample(frac=n_sample / len(df_customers), random_state=42))
)

# 3. Systematic sampling (every k-th customer)
k = len(df_customers) // n_sample
sample_systematic = df_customers.iloc[::k].head(n_sample)

# Compare estimates
pop_mean = df_customers['avg_ticket'].mean()
print('=' * 55)
print('SAMPLING COMPARISON — avg_ticket estimate')
print('=' * 55)
print(f'Census mean (truth)       : ${pop_mean:>10,.0f} CLP')
print(f'Simple random sample mean : ${sample_random["avg_ticket"].mean():>10,.0f} CLP  (n={len(sample_random)})')
print(f'Stratified sample mean    : ${sample_stratified["avg_ticket"].mean():>10,.0f} CLP  (n={len(sample_stratified)})')
print(f'Systematic sample mean    : ${sample_systematic["avg_ticket"].mean():>10,.0f} CLP  (n={len(sample_systematic)})')
print()
print('Lean conclusion: stratified sampling preserves segment')
print('proportions and produces the closest estimate to the census mean.')

---
## 8. Probability Summary — Connection to Hypotheses

| Probability | Value | Hypothesis Connected |
|-------------|-------|---------------------|
| P(Dormant) = churn rate | *Run cells above* | H3 — churn vs 30% benchmark |
| P(Promoter) | *Run cells above* | H4 — NPS effect on avg_ticket |
| P(Promoter \| Active) | *Run cells above* | H4 — conditional NPS behavior |
| P(Dormant \| Detractor) | *Run cells above* | H3 + H4 — combined risk signal |
| P(NPS \| Segment) — full tree | *See heatmap* | H4 — ANOVA grouping validation |

> **Key insight:** P(Dormant | Detractor) >> P(Dormant) confirms that
> NPS category is a leading indicator of churn — supporting the relevance
> of H4 and the business case for NPS-based intervention.


---
## 9. LEAN Filter — Waste Elimination Review

| LEAN Question | Answer | Action |
|---------------|--------|--------|
| Does every probability connect to a hypothesis? | ✅ Yes — all calculations map to H1–H4 | Proceed |
| Did we calculate unnecessary probabilities? | ✅ No — excluded region, product, season probabilities (not in hypotheses) | No waste |
| Is the sampling demo adding value? | ✅ Yes — required by M5 consigna + demonstrates methodological awareness | Keep |
| Is the probability tree interpretable? | ✅ Yes — heatmap + stacked bar gives clear signal | Proceed |
| Key insight generated? | ✅ Yes — P(Dormant \| Detractor) >> P(Dormant) is actionable | High value |


---
## 10. Decisions Log — Lesson 2

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|----------|-----------|------------------------|-------------|
| 1 | Use `retargeting_segment` as primary event space | Directly maps to churn (H3) and business segments | Platform or region | ✅ Highest business relevance |
| 2 | Calculate P(NPS \| Segment) not P(Segment \| NPS) | Segment is the actionable lever — NPS is the outcome | Reverse direction | ✅ Correct causal direction |
| 3 | Demonstrate 3 sampling methods even though we have a census | Required by M5 consigna; also shows methodological awareness | Skip — we have census | ✅ Portfolio + compliance value |
| 4 | Use n=100 for sampling demo | Minimum required by consigna | n=200 | ✅ MVA principle |


---
## 11. Next Steps — Lesson 3 Preview

In **Lesson 3 — Data Preparation (notebook 03)**, the following tasks will be completed:

1. Identify whether `avg_ticket` follows a Normal distribution (Shapiro-Wilk, Q-Q plot)
2. Test whether `total_transactions` follows a Poisson distribution
3. Model churn as a Binomial variable and calculate probabilities
4. Justify distribution choices for each hypothesis test
5. Plot all distributions with overlaid theoretical curves
6. Apply LEAN filter: does each distribution justify the planned test?

---

**Previous:** [01 - Business Understanding](./01_business_understanding.ipynb)  
**Next Phase -->** [03 - Data Preparation](./03_data_preparation.ipynb)
